# Cryptocurrency Price Manipulation Detection
**Author:** Dharmateja PONNAM | **Student ID:** 20240386

Detects potential pump-and-dump and other manipulation patterns in crypto markets using Isolation Forest anomaly detection on Binance OHLCV data.

## 0. Setup

In [ ]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.append('..')  # so Python can find the src/ package

import matplotlib.pyplot as plt
import pandas as pd

from src.data_fetch import fetch_multiple_cryptos, fetch_historical_data, save_to_csv
from src.features import load_and_clean, engineer_features, detect_pump_events
from src.detection import train_isolation_forest, get_anomalies

## 1. Fetch Data

In [ ]:
SYMBOLS = ["BTC/USDT", "ETH/USDT", "BNB/USDT", "XRP/USDT", "ADA/USDT"]

# Live snapshot
live_data = fetch_multiple_cryptos(SYMBOLS)
live_data

In [ ]:
# Historical OHLCV — last 100 hourly candles
historical_data = fetch_historical_data(SYMBOLS, timeframe="1h", limit=100)

# Save to data/raw/
save_to_csv(historical_data, output_dir="../data/raw")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Load and clean BTC data
btc_data = load_and_clean("../data/raw/BTC_USDT_data.csv")
btc_data.head()

In [ ]:
# Bitcoin closing price over time
plt.figure(figsize=(12, 6))
plt.plot(btc_data["timestamp"], btc_data["close"], label="BTC Price", color="blue")
plt.xlabel("Time")
plt.ylabel("Price (USDT)")
plt.title("Bitcoin Price Over Time")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/plots/btc_price_over_time.png", dpi=150)
plt.show()

## 3. Feature Engineering & Pump Detection

In [ ]:
# Add price_change, volume_change, volatility
btc_data = engineer_features(btc_data)
btc_data.head()

In [ ]:
# Threshold-based pump event detection (>5% price rise in one candle)
pump_events = detect_pump_events(btc_data, threshold=5.0)
pump_events

In [ ]:
# Visualise pump events
plt.figure(figsize=(12, 6))
plt.plot(btc_data["timestamp"], btc_data["close"], label="BTC Price", color="blue")
plt.scatter(pump_events["timestamp"], pump_events["close"],
            color="red", label="Pump Detected", marker="o")
plt.xlabel("Time")
plt.ylabel("Price (USDT)")
plt.title("Bitcoin Pump Events")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/plots/btc_pump_events.png", dpi=150)
plt.show()

## 4. Anomaly Detection — Isolation Forest

In [ ]:
# Train model and label anomalies
btc_data, model = train_isolation_forest(btc_data, contamination=0.01)

# Extract anomalous rows
anomalies = get_anomalies(btc_data)
anomalies

In [ ]:
# Visualise anomalies
plt.figure(figsize=(12, 6))
plt.plot(btc_data["timestamp"], btc_data["close"], label="BTC Price", color="blue")
plt.scatter(anomalies["timestamp"], anomalies["close"],
            color="red", label="Anomaly Detected", marker="o")
plt.xlabel("Time")
plt.ylabel("Price (USDT)")
plt.title("Bitcoin Anomalies (Pump-and-Dump Detection)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/plots/btc_anomalies.png", dpi=150)
plt.show()

## 5. Results

| Metric | Value |
|---|---|
| Total candles analysed | `len(btc_data)` |
| Anomalies flagged | `len(anomalies)` |
| Contamination rate | 1% |

Most anomalies coincide with sudden price spikes and volume surges — consistent with pump-and-dump behaviour.